In [2]:
# Exact Match & Subset Duplication (Customer Orders Cleanup)
# The Problem: You receive an e-commerce transaction dataset where users accidentally clicked the "Submit Order" button twice,
# creating exact duplicate rows. However, you only want to consider the CustomerID and OrderDate columns to identify duplicates,
# ignoring minor system-generated log IDs.

import pandas as pd

# Sample dataset with exact and partial duplicate rows

data = {
    'CustomerID': [101, 102, 101, 103, 102],
    'OrderDate': ['2026-06-01', '2026-06-02', '2026-06-01', '2026-06-03', '2026-06-02'],
    'Amount': [250, 400, 250, 150, 420], # Note: Amount might slightly differ in a partial duplicate case
    'Status': ['Completed', 'Pending', 'Completed', 'Completed', 'Pending']
}

df = pd.DataFrame(data)
print("Original DataFrame:\n", df)

# 1. Detect exact duplicates across all columns

exact_dups = df[df.duplicated(keep=False)]
print("\nExact Duplicates:\n", exact_dups)

# 2. Drop duplicates based on specific business keys (subset)
# Keeping the first occurrence and dropping subsequent ones

cleaned_df = df.drop_duplicates(subset=['CustomerID', 'OrderDate'], keep='first')
print("\nCleaned DataFrame after subset deduplication:\n", cleaned_df)

Original DataFrame:
    CustomerID   OrderDate  Amount     Status
0         101  2026-06-01     250  Completed
1         102  2026-06-02     400    Pending
2         101  2026-06-01     250  Completed
3         103  2026-06-03     150  Completed
4         102  2026-06-02     420    Pending

Exact Duplicates:
    CustomerID   OrderDate  Amount     Status
0         101  2026-06-01     250  Completed
2         101  2026-06-01     250  Completed

Cleaned DataFrame after subset deduplication:
    CustomerID   OrderDate  Amount     Status
0         101  2026-06-01     250  Completed
1         102  2026-06-02     400    Pending
3         103  2026-06-03     150  Completed


In [8]:
# Fuzzy Matching for Text Data (Customer Name Standardization)
# The Problem: You are analyzing a CRM database, but user names have typos,
# missing middle names, or slight spelling variations entered by different operators (e.g., "John Smith" vs. "Jon Smith" or "John Smitt").
# Eexact matching will fail entirely here.

import pandas as pd
from thefuzz import fuzz

# Sample dirty customer list

df_names = pd.DataFrame({
    'ID': [1, 2, 3, 4],
    'RawName': ['John Smith', 'Jon Smith', 'Alice Johnson', 'John Smitt']
})

# Let's check similarity scores using thefuzz library

name_1 = df_names.loc[0, 'RawName']
name_2 = df_names.loc[1, 'RawName']
name_3 = df_names.loc[3, 'RawName']

print(f"Similarity between '{name_1}' and '{name_2}': {fuzz.ratio(name_1, name_2)}%")
print(f"Similarity between '{name_1}' and '{name_3}': {fuzz.ratio(name_1, name_3)}%")

# Practical Workflow Tip:
# In a real project, you iterate through pairs, calculate similarity, 
# and flag records with a ratio > 80 as potential duplicates for review or automatic merging.

Similarity between 'John Smith' and 'Jon Smith': 95%
Similarity between 'John Smith' and 'John Smitt': 90%


In [ ]:
# Time-Based Log Deduplication (Handling User Session Logs)
# The Problem: In product analytics or web tracking, 
# user event logs often contain multiple rapid actions
# (e.g., a user clicking a login button multiple times within seconds or logging in multiple times a day).
# If you are analyzing daily active users, you need to sort the timestamps and keep only the latest
# or first session record per user instead of dropping rows randomly.

import pandas as pd

# Sample user login event logs with timestamps

data = {
    'UserID': [101, 102, 101, 103, 101],
    'LoginTime': ['2026-10-26 08:30:00', '2026-10-26 09:15:00', '2026-10-26 08:31:00', '2026-10-26 10:00:00', '2026-10-26 11:00:00'],
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Tablet', 'Desktop']
}
df = pd.DataFrame(data)

# Convert string timestamps to datetime objects for accurate sorting

df['LoginTime'] = pd.to_datetime(df['LoginTime'])

# 1. Sort values by UserID and LoginTime to chronological order

df_sorted = df.sort_values(by=['UserID', 'LoginTime'])

# 2. Drop duplicates keeping the LAST login session for each user

latest_sessions = df_sorted.drop_duplicates(subset=['UserID'], keep='last')

print("Latest User Sessions:\n", latest_sessions)

Latest User Sessions:
    UserID           LoginTime   Device
4     101 2026-10-26 11:00:00  Desktop
1     102 2026-10-26 09:15:00  Desktop
3     103 2026-10-26 10:00:00   Tablet


In [12]:
df_sorted

,UserID,LoginTime,Device
0,101,2026-10-26 08:30:00,Mobile
2,101,2026-10-26 08:31:00,Mobile
4,101,2026-10-26 11:00:00,Desktop
1,102,2026-10-26 09:15:00,Desktop
3,103,2026-10-26 10:00:00,Tablet


In [10]:
# Cross-Column Redundancy & Data Integrity Check (Schema Audit)
# The Problem: In corporate databases, 
# you often encounter redundant columns (e.g., separate FirstName and LastName columns alongside a combined FullName column).
# Over time, updates happen asynchronously, causing data inconsistencies where FullName doesn't match the actual components.
# This damages data integrity and introduces silent bugs into your machine learning feature engineering pipeline.

import pandas as pd

# Sample employee dataset with redundant/inconsistent columns
data = {
    'EmpID': [1, 2, 3],
    'FirstName': ['John', 'Alice', 'Bob'],
    'LastName': ['Smith', 'Johnson', 'Brown'],
    'FullName': ['John Smith', 'Alice Smith', 'Bob Brown'] # Notice row 2 has a typo/inconsistency ('Alice Smith' vs 'Alice Johnson')
}

df = pd.DataFrame(data)

# 1. Reconstruct the expected FullName from individual columns

df['ExpectedFullName'] = df['FirstName'] + ' ' + df['LastName']

# 2. Detect data integrity issues (where records are out of sync)

inconsistencies = df[df['FullName'] != df['ExpectedFullName']]
print("Data Integrity Discrepancies Found:\n", inconsistencies)

# 3. Clean up: Drop the unreliable redundant column and keep the verified one

cleaned_df = df.drop(columns=['FullName']).rename(columns={'ExpectedFullName': 'FullName'})
print("\nCleaned and Verified DataFrame:\n", cleaned_df)

Data Integrity Discrepancies Found:
    EmpID FirstName LastName     FullName ExpectedFullName
1      2     Alice  Johnson  Alice Smith    Alice Johnson

Cleaned and Verified DataFrame:
    EmpID FirstName LastName       FullName
0      1      John    Smith     John Smith
1      2     Alice  Johnson  Alice Johnson
2      3       Bob    Brown      Bob Brown


In [13]:
import pandas as pd

# 1. Create a sample dataset simulating an e-commerce order table
# Some users accidentally clicked "Submit Order" twice, creating duplicates.

data = {
    'OrderID': [1001, 1002, 1003, 1001, 1004, 1002],
    'CustomerID': [501, 502, 503, 501, 504, 502],
    'OrderAmount': [150.0, 300.0, 450.0, 150.0, 120.0, 300.0],
    'OrderDate': ['2026-10-01', '2026-10-01', '2026-10-02', '2026-10-01', '2026-10-02', '2026-10-01']
}

df = pd.DataFrame(data)
print("--- Original DataFrame ---")
print(df)

# 2. Audit & Detect: Use keep=False to isolate and inspect all duplicate rows

duplicates_audit = df[df.duplicated(subset=['OrderID', 'CustomerID'], keep=False)]
print("\n--- Auditing Duplicates Found ---")
print(duplicates_audit)

# 3. Clean: Drop duplicates based on business keys, keeping the first valid transaction

cleaned_df = df.drop_duplicates(subset=['OrderID', 'CustomerID'], keep='first').reset_index(drop=True)
print("\n--- Cleaned DataFrame ---")
print(cleaned_df)

# 4. Business Impact Check (Why this matters for Data Science)

original_revenue = df['OrderAmount'].sum()
cleaned_revenue = cleaned_df['OrderAmount'].sum()

print("\n--- Impact Analysis ---")
print(f"Original Revenue (with duplicates): ${original_revenue}")
print(f"Actual Revenue (after cleaning): ${cleaned_revenue}")
print(f"Inflated Error Amount: ${original_revenue - cleaned_revenue}")

--- Original DataFrame ---
   OrderID  CustomerID  OrderAmount   OrderDate
0     1001         501        150.0  2026-10-01
1     1002         502        300.0  2026-10-01
2     1003         503        450.0  2026-10-02
3     1001         501        150.0  2026-10-01
4     1004         504        120.0  2026-10-02
5     1002         502        300.0  2026-10-01

--- Auditing Duplicates Found ---
   OrderID  CustomerID  OrderAmount   OrderDate
0     1001         501        150.0  2026-10-01
1     1002         502        300.0  2026-10-01
3     1001         501        150.0  2026-10-01
5     1002         502        300.0  2026-10-01

--- Cleaned DataFrame ---
   OrderID  CustomerID  OrderAmount   OrderDate
0     1001         501        150.0  2026-10-01
1     1002         502        300.0  2026-10-01
2     1003         503        450.0  2026-10-02
3     1004         504        120.0  2026-10-02

--- Impact Analysis ---
Original Revenue (with duplicates): $1470.0
Actual Revenue (after clea

In [14]:
import pandas as pd
from thefuzz import fuzz

# 1. Create a sample CRM dataset with spelling variations and typos

crm_data = {
    'UserID': [1, 2, 3, 4, 5],
    'FullName': ['Ali Rezaei', 'Ali Rezaii', 'Sara Movahed', 'Ali Alizadeh', 'Sarah Movahed']
}

df_crm = pd.DataFrame(crm_data)
print("--- Original CRM Database ---")
print(df_crm)

# 2. Demonstrate Fuzzy Matching between a target name and others

target_name = df_crm.loc[0, 'FullName']
print(f"\nComparing names against target: '{target_name}'\n")

for idx, row in df_crm.iterrows():
    
    # fuzz.ratio calculates the similarity percentage between two strings
    
    similarity_score = fuzz.ratio(target_name, row['FullName'])
    print(f"- Compare with '{row.FullName}': Similarity = {similarity_score}%")

# 3. Automated Threshold Filtering to flag potential duplicates for review

threshold = 80
potential_duplicates = []

for i in range(len(df_crm)):
    for j in range(i + 1, len(df_crm)):
        name1 = df_crm.loc[i, 'FullName']
        name2 = df_crm.loc[j, 'FullName']
        score = fuzz.ratio(name1, name2)
        
        if score >= threshold:
            potential_duplicates.append({
                'Record_1': name1,
                'Record_2': name2,
                'Similarity_Score': score
            })

# Convert results into a DataFrame for clean reporting

dup_report_df = pd.DataFrame(potential_duplicates)
print("\n--- Potential Duplicate Report (Score >= 80%) ---")
print(dup_report_df)

--- Original CRM Database ---
   UserID       FullName
0       1     Ali Rezaei
1       2     Ali Rezaii
2       3   Sara Movahed
3       4   Ali Alizadeh
4       5  Sarah Movahed

Comparing names against target: 'Ali Rezaei'

- Compare with 'Ali Rezaei': Similarity = 100%
- Compare with 'Ali Rezaii': Similarity = 90%
- Compare with 'Sara Movahed': Similarity = 27%
- Compare with 'Ali Alizadeh': Similarity = 64%
- Compare with 'Sarah Movahed': Similarity = 26%

--- Potential Duplicate Report (Score >= 80%) ---
       Record_1       Record_2  Similarity_Score
0    Ali Rezaei     Ali Rezaii                90
1  Sara Movahed  Sarah Movahed                96
